# Uma Rede Neural Simples do Zero com PyTorch

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


Neste tutorial implementamos uma rede neural simples do zero usando PyTorch.


## Sobre

Neste tutorial vamos implementar uma rede neural simples do zero usando PyTorch. A ideia é ensinar o básico do PyTorch e como ele pode ser usado para construir uma rede neural manualmente. Vou cobrir algumas funcionalidades e conceitos básicos disponíveis no PyTorch.

O tutorial assume conhecimento prévio sobre redes neurais. Se você não tem certeza, ainda assim deve conseguir acompanhar. Para usuários avançados, este tutorial serve como revisão.

O módulo `torch` provê todos os operadores **tensor** necessários para implementar sua primeira rede neural do zero em PyTorch. Em PyTorch tudo é Tensor — é a primeira coisa a se acostumar. Vamos importar as bibliotecas necessárias:


In [ ]:
import torch
import torch.nn as nn

## Dados

Vamos começar criando alguns dados de exemplo usando o comando `torch.tensor`. No NumPy faríamos com `np.array`. Em PyTorch, tudo é Tensor (não vetor ou matriz). Definimos os tipos com `dtype=torch.xxx`.

Nos dados abaixo, `X` representa a quantidade de horas estudadas e o tempo dormido; `y` representa as notas. A variável `xPredicted` é uma única entrada para a qual queremos prever a nota usando os parâmetros aprendidos pela rede.


In [ ]:
X = torch.tensor(([2, 9], [1, 5], [3, 6]), dtype=torch.float) # 3 X 2 tensor
y = torch.tensor(([92], [100], [89]), dtype=torch.float) # 3 X 1 tensor
xPredicted = torch.tensor(([4, 8]), dtype=torch.float) # 1 X 2 tensor

Você pode checar o tamanho dos tensores recém-criados com `size()`. É equivalente a `shape` em NumPy / TensorFlow.


In [ ]:
print(X.size())
print(y.size())

## Escalonamento

Abaixo aplicamos escalonamento nos dados. Note que a função `max` retorna tanto o tensor quanto os índices correspondentes. Usamos `_` para descartar os índices, já que aqui só queremos os valores máximos. Os dados ficam agora num formato bem amigável para a rede.


In [ ]:
# scale units
X_max, _ = torch.max(X, 0)
xPredicted_max, _ = torch.max(xPredicted, 0)

X = torch.div(X, X_max)
xPredicted = torch.div(xPredicted, xPredicted_max)
y = y / 100  # max test score is 100
print(xPredicted)

## Modelo (Grafo Computacional)

Com os dados processados e no formato certo, falta definir o modelo. Aqui é onde as coisas mudam um pouco em relação a Keras / TensorFlow, mas no fim das contas estamos construindo um grafo computacional que dita o fluxo de dados e as operações aplicadas.

Para fins ilustrativos, vamos construir a seguinte rede neural / grafo computacional:


In [ ]:
class Neural_Network(nn.Module):
    def __init__(self, ):
        super(Neural_Network, self).__init__()
        # parameters
        # TODO: parameters can be parameterized instead of declaring them here
        self.inputSize = 2
        self.outputSize = 1
        self.hiddenSize = 3
        
        # weights
        self.W1 = torch.randn(self.inputSize, self.hiddenSize) # 3 X 2 tensor
        self.W2 = torch.randn(self.hiddenSize, self.outputSize) # 3 X 1 tensor
        
    def forward(self, X):
        self.z = torch.matmul(X, self.W1) # 3 X 3 ".dot" does not broadcast in PyTorch
        self.z2 = self.sigmoid(self.z) # activation function
        self.z3 = torch.matmul(self.z2, self.W2)
        o = self.sigmoid(self.z3) # final activation function
        return o
        
    def sigmoid(self, s):
        return 1 / (1 + torch.exp(-s))
    
    def sigmoidPrime(self, s):
        # derivative of sigmoid
        return s * (1 - s)
    
    def backward(self, X, y, o):
        self.o_error = y - o # error in output
        self.o_delta = self.o_error * self.sigmoidPrime(o) # derivative of sig to error
        self.z2_error = torch.matmul(self.o_delta, torch.t(self.W2))
        self.z2_delta = self.z2_error * self.sigmoidPrime(self.z2)
        self.W1 += torch.matmul(torch.t(X), self.z2_delta)
        self.W2 += torch.matmul(torch.t(self.z2), self.o_delta)
        
    def train(self, X, y):
        # forward + backward pass for training
        o = self.forward(X)
        self.backward(X, y, o)
        
    def saveWeights(self, model):
        # we will use the PyTorch internal storage functions
        torch.save(model, "NN")
        # you can reload model with all the weights and so forth with:
        # torch.load("NN")
        
    def predict(self):
        print ("Predicted data based on trained weights: ")
        print ("Input (scaled): \n" + str(xPredicted))
        print ("Output: \n" + str(self.forward(xPredicted)))
        

Neste tutorial não vou entrar na matemática — fica para outro dia. Quero só dar a essência de como construir uma rede neural do zero usando PyTorch. Vamos destrinchar o modelo declarado acima.

## Cabeçalho da Classe

Definimos o modelo via classe, jeito recomendado de construir um grafo computacional. O cabeçalho contém o nome da classe (`Neural_Network`) e herda de `nn.Module`, indicando que estamos definindo nossa própria rede:

```python
class Neural_Network(nn.Module):
```

## Inicialização

Em `__init__`, definimos inicializações executadas ao criar a instância. Tipicamente declaramos aqui a estrutura da rede — tamanho das camadas ocultas etc. Como estamos fazendo do zero, declaramos explicitamente o tamanho das matrizes de pesos: uma da entrada para a camada oculta, outra da oculta para a saída. Os pesos são amostrados de uma normal via `torch.randn(...)`. Para manter simples, não usamos bias.

```python
def __init__(self):
    super().__init__()
    self.inputSize = 2
    self.outputSize = 1
    self.hiddenSize = 3
    self.W1 = torch.randn(self.inputSize, self.hiddenSize)  # 2 x 3
    self.W2 = torch.randn(self.hiddenSize, self.outputSize) # 3 x 1
```

## Função Forward

A função `forward` é onde a mágica acontece: os dados entram e fluem pelo grafo. Como temos uma única camada oculta, fica simples:

```python
def forward(self, X):
    self.z = torch.matmul(X, self.W1)
    self.z2 = self.sigmoid(self.z)         # ativação
    self.z3 = torch.matmul(self.z2, self.W2)
    return self.sigmoid(self.z3)
```

A função recebe `X`, multiplica pela matriz `self.W1`, aplica sigmoide, multiplica por `self.W2`, e aplica sigmoide de novo. Esse é o feed-forward. Para otimizar os pesos, precisamos do algoritmo de retropropagação.

## Função Backward

A função `backward` contém o algoritmo de retropropagação, cujo objetivo é minimizar a perda em relação aos pesos:

```python
def backward(self, X, y, o):
    self.o_error = y - o
    self.o_delta = self.o_error * self.sigmoidPrime(o)
    self.z2_error = torch.matmul(self.o_delta, torch.t(self.W2))
    self.z2_delta = self.z2_error * self.sigmoidPrime(self.z2)
    self.W1 += torch.matmul(torch.t(X), self.z2_delta)
    self.W2 += torch.matmul(torch.t(self.z2), self.o_delta)
```

Várias multiplicações de matriz e transposições — o resto é simplesmente gradiente descendente.


## Treinamento

Resta treinar a rede. Primeiro criamos a instância:

```python
NN = Neural_Network()
```

Treinamos por `1000` épocas. Em PyTorch, `NN(X)` chama automaticamente o `forward`, então não precisamos invocar `NN.forward(X)` explicitamente.

Calculamos a perda a cada época com:

```python
torch.mean((y - NN(X))**2).detach().item()
```

O próximo passo é treinar (forward + backward) via `NN.treinar(X, y)`. Depois de treinada, podemos salvar o modelo e gerar a predição para a entrada `xPredicted` declarada no início.

Vamos treinar!


In [ ]:
NN = Neural_Network()
for i in range(1000):  # trains the NN 1,000 times
    if (i % 100) == 0:
        print ("#" + str(i) + " Loss: " + str(torch.mean((y - NN(X))**2).detach().item()))  # mean sum squared loss
    NN.train(X, y)
#NN.saveWeights(NN) # save weights

NN.predict()

print("Finished training!")

## Referências

- [PyTorch — `nn.Module`](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html#pytorch-custom-nn-modules)


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
